In [97]:
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path


from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold

### Add any other import statements you need below

**government spending per month**

In [12]:
pd.read_excel(r'ds-government-spending/MonthlyReceiptsOutlaysDeficitSurplus.xls').head(5)

,Period,Receipts,Outlays,Deficit/Surplus (-),Borrowing from the Public,Reduction of Operating Cash,By Other Means
0,1980-10-31 00:00:00,38923,55843,16921,NaN,NaN,NaN
1,1980-11-30 00:00:00,39175,48083,8907,NaN,NaN,NaN
2,1980-12-31 00:00:00,48904,50869,1965,NaN,NaN,NaN
3,1981-01-31 00:00:00,51147,63339,12192,NaN,NaN,NaN
4,1981-02-28 00:00:00,38129,53749,15620,NaN,NaN,NaN


In [13]:
pd.read_excel(r'ds-government-spending/cpi-u-202512.xlsx', header=4).head(5)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Dec.\n2024,Jan.\n2025,Feb.\n2025,Mar.\n2025,Apr.\n2025,May\n2025,Jun.\n2025,...,Oct.\n2025-\nNov.\n2025.1,Nov.\n2025-\nDec.\n2025.1,Seasonally adjusted effect on All Items\nNov. 2025-\nDec. 2025(1),"Standard error, median price change(2)",Largest (L) or Smallest (S) seasonally adjusted change since:(3),Largest (L) or Smallest (S) seasonally adjusted change since:(3).1,Unadjusted effect on All Items\nDec. 2024-\nDec. 2025(1),"Standard error, median price change(2).1",Largest (L) or Smallest (S) unadjusted change since:(3),Largest (L) or Smallest (S) unadjusted change since:(3).1
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Date,Percent change,NaN,NaN,Date,Percent change
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,All items,100,315.605,317.671,319.082,319.799,320.795,321.465,322.561,...,–,0.3,–,0.05,–,–,–,0.11,–,–
3,1.0,Food,13.657,333.566,335.517,336.274,337.751,337.747,338.598,339.498,...,–,0.7,0.097,0.07,L-Oct. 2022,0.7,0.42,0.13,L-Sep. 2025,3.1
4,2.0,Food at home,7.973,308.38,310.936,311.284,312.815,311.84,312.491,313.028,...,–,0.7,0.058,0.12,L-Aug. 2022,0.9,0.19,0.18,L-Sep. 2025,2.7


In [98]:
#trying to make it prettier by moving the header
file = "ds-government-spending/cpi-u-202512.xlsx"

months = pd.date_range(
    "2025-01-01", periods=12, freq="MS"
).strftime("%b %Y").tolist()

df = pd.read_excel(
    file,
    sheet_name="US",
    header=None,
    skiprows=7,
    usecols="A:B,E:P"
)

df.columns = ["Level", "Category", *months]

df = df[pd.to_numeric(df["Level"], errors="coerce").notna()]
df = df.drop(columns="Level")


df.head(5)

,Category,Jan 2025,Feb 2025,Mar 2025,Apr 2025,May 2025,Jun 2025,Jul 2025,Aug 2025,Sep 2025,Oct 2025,Nov 2025,Dec 2025
0,All items,317.671,319.082,319.799,320.795,321.465,322.561,323.048,323.976,324.8,–,324.122,324.054
1,Food,335.517,336.274,337.751,337.747,338.598,339.498,340.036,341.295,342.405,–,341.721,343.795
2,Food at home,310.936,311.284,312.815,311.84,312.491,313.028,313.263,314.608,316.042,–,313.954,315.664
3,Cereals and bakery products,355.964,357.627,358.45,355.976,360.325,360.04,360.048,359.74,361.894,–,361.255,361.984
4,Cereals and cereal products,282.3,285.624,287.383,282.614,287.38,285.075,283.665,282.729,283.727,–,287.875,286.9


In [56]:
#obverving egg prices 2024 -2025
df = pd.read_excel(file, sheet_name="US", header=None)

eggs = pd.DataFrame({
    "Month": df.iloc[4, 3:16].str.replace("\n", " "),
    "Egg CPI": pd.to_numeric(df.iloc[57, 3:16], errors="coerce")
})

eggs = eggs.reset_index(drop=True)

alt.Chart(eggs).mark_line(point=True).encode(
    x=alt.X("Month:O", sort=None),
    y=alt.Y("Egg CPI:Q"),
    tooltip=["Month", "Egg CPI"]
).properties(
    title="Egg CPI 2024 - 2025",
)

alt.Chart(...)

**Baby Name trend process**

In [86]:
url = "myanh-test/babyname-top5-2025.html"
#read in as python
babyname_2025 = pd.read_html(url)

#how many tables are there
len(babyname_2025)

2

In [87]:
#girl table name
girls_2025= babyname_2025[0]

girls_2025.head()

,State,Rank 1,Rank 2,Rank 3,Rank 4,Rank 5
0,Alabama,Charlotte,Amelia,Olivia,Elizabeth,Emma
1,Alaska,Charlotte,Amelia,Emma,Evelyn,Aurora
2,Arizona,Olivia,Sophia,Mia,Amelia,Isabella
3,Arkansas,Charlotte,Amelia,Olivia,Isabella,Eleanor
4,California,Olivia,Mia,Sophia,Camila,Emma


In [88]:
#boys table name
boys_2025 = babyname_2025[1]

boys_2025.head()

,State,Rank 1,Rank 2,Rank 3,Rank 4,Rank 5
0,Alabama,John,William,James,Liam,Noah
1,Alaska,Noah,Theodore,Henry,Oliver,James
2,Arizona,Liam,Noah,Mateo,Santiago,Oliver
3,Arkansas,Liam,Hudson,Oliver,Elijah,Noah
4,California,Liam,Noah,Mateo,Santiago,Sebastian


In [99]:
#read in other years

folder = Path("/Users/myanh/Documents/DATA3301/myanh-test")

all_names = []

for file in sorted(folder.glob("babyname-top5-*.html")):
    year = int(file.stem.split("-")[-1])
    tables = pd.read_html(file)

    for sex, table in zip(["F", "M"], tables):
        names = table.melt(
            id_vars="State",
            var_name="rank",
            value_name="name"
        )

        names["rank"] = names["rank"].str.replace("Rank ", "").astype(int)
        names["year"] = year
        names["sex"] = sex

        all_names.append(names)

baby_names = pd.concat(all_names, ignore_index=True)

baby_names.head()

,State,rank,name,year,sex
0,Alabama,1,Olivia,2022,F
1,Alaska,1,Aurora,2022,F
2,Arizona,1,Olivia,2022,F
3,Arkansas,1,Olivia,2022,F
4,California,1,Olivia,2022,F


In [ ]:
#checking the olivia name through the year
olivia = baby_names[baby_names["name"] == "Olivia"]

olivia_count = (
    olivia.groupby("year")["State"]
    .nunique()
    .reset_index(name="states")
)

chart = alt.Chart(olivia_count).mark_line(point=True).encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("states:Q", title="Number of States"),
    tooltip=["year", "states"]
).properties(
    title="Olivia's Popularity Across States"
)

chart

alt.Chart(...)

**Government spending**

2024 highway disbursements by state

In [89]:
#read in html files and check how many tables 
url = "myanh-test/Table HF-2 - Policy | Federal Highway Administration.html"
highway_fund_py = pd.read_html(url)

len(highway_fund_py)

1

In [90]:
highway_fund = highway_fund_py[0]
highway_fund.info()

<class 'pandas.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 14 columns):
 #   Column                                                                            Non-Null Count  Dtype
---  ------                                                                            --------------  -----
 0   (STATE, STATE)                                                                    53 non-null     str  
 1   (CAPITAL OUTLAY, STATE ADMINISTERED HIGHWAYS)                                     53 non-null     str  
 2   (CAPITAL OUTLAY, LOCALLY ADMINISTERED ROADS)                                      53 non-null     str  
 3   (CAPITAL OUTLAY, FEDERAL ROADS AND UNCLASSIFIED)                                  53 non-null     str  
 4   (CAPITAL OUTLAY, TOTAL)                                                           53 non-null     int64
 5   (MAINTENANCE AND SERVICES, STATE ADMINISTERED HIGHWAYS)                           53 non-null     str  
 6   (MAINTENANCE AND SERVICES, LOCA

In [91]:
highway_fund.head()

STATE              CAPITAL OUTLAY                             \
        STATE STATE ADMINISTERED HIGHWAYS LOCALLY ADMINISTERED ROADS   
0     Alabama                     1082377                     532245   
1      Alaska                      862632                     125923   
2     Arizona                     1248888                     798850   
3    Arkansas                     1602089                     226569   
4  California                     6974319                    5653622   

                                              MAINTENANCE AND SERVICES  \
  FEDERAL ROADS AND UNCLASSIFIED     TOTAL STATE ADMINISTERED HIGHWAYS   
0                             16   1614638                       39663   
1                              -    988555                      156559   
2                              -   2047738                      185896   
3                            839   1829497                      231872   
4                              -  12627941                     2758204   

                                                                      \
  LOCALLY ADMINISTERED ROADS FEDERAL ROADS AND UNCLASSIFIED    TOTAL   
0                     444070                              -   483733   
1                     167313                              -   323872   
2                     823197                              -  1009093   
3                     405974                              -   637846   
4                    5112729                              -  7870933   

  ADMIN-  ISTRATION AND MISCEL-  LANEOUS HIGHWAY LAW ENFORCEMENT AND SAFETY  \
  ADMIN-  ISTRATION AND MISCEL-  LANEOUS HIGHWAY LAW ENFORCEMENT AND SAFETY   
0                                 558335                             262651   
1                                 106047                             142171   
2                                 814306                             202285   
3                                 387038                             329322   
4                                1213033                            5361640   

  INTEREST BOND RETIREMENT (2) TOTAL DISBURSE-  MENTS  
  INTEREST BOND RETIREMENT (2) TOTAL DISBURSE-  MENTS  
0    16130              113943                3049430  
1    11591               29427                1601663  
2   111227              610681                4795330  
3    19990              128860                3332553  
4  1137770             1444774               29656091

In [92]:
#checking data types
highway_fund.dtypes

STATE                                   STATE                                       str
CAPITAL OUTLAY                          STATE ADMINISTERED HIGHWAYS                 str
                                        LOCALLY ADMINISTERED ROADS                  str
                                        FEDERAL ROADS AND UNCLASSIFIED              str
                                        TOTAL                                     int64
MAINTENANCE AND SERVICES                STATE ADMINISTERED HIGHWAYS                 str
                                        LOCALLY ADMINISTERED ROADS                  str
                                        FEDERAL ROADS AND UNCLASSIFIED              str
                                        TOTAL                                     int64
ADMIN-  ISTRATION AND MISCEL-  LANEOUS  ADMIN-  ISTRATION AND MISCEL-  LANEOUS    int64
HIGHWAY LAW ENFORCEMENT AND SAFETY      HIGHWAY LAW ENFORCEMENT AND SAFETY          str
INTEREST                        

In [93]:
highway_fund["INTEREST"].head()

,INTEREST
0,16130
1,11591
2,111227
3,19990
4,1137770


In [94]:
#converting data type to numeric 
for col in highway_fund.columns:
    if col[0] != "STATE":
        highway_fund[col] = pd.to_numeric(
            highway_fund[col],
            errors="coerce"
        )

highway_fund.dtypes

STATE                                   STATE                                         str
CAPITAL OUTLAY                          STATE ADMINISTERED HIGHWAYS               float64
                                        LOCALLY ADMINISTERED ROADS                float64
                                        FEDERAL ROADS AND UNCLASSIFIED            float64
                                        TOTAL                                       int64
MAINTENANCE AND SERVICES                STATE ADMINISTERED HIGHWAYS               float64
                                        LOCALLY ADMINISTERED ROADS                float64
                                        FEDERAL ROADS AND UNCLASSIFIED            float64
                                        TOTAL                                       int64
ADMIN-  ISTRATION AND MISCEL-  LANEOUS  ADMIN-  ISTRATION AND MISCEL-  LANEOUS      int64
HIGHWAY LAW ENFORCEMENT AND SAFETY      HIGHWAY LAW ENFORCEMENT AND SAFETY        float64
INTEREST  

In [95]:
highway_fund.columns = highway_fund.columns.droplevel(0)
highway_fund.head()

,STATE,STATE ADMINISTERED HIGHWAYS,LOCALLY ADMINISTERED ROADS,FEDERAL ROADS AND UNCLASSIFIED,TOTAL,STATE ADMINISTERED HIGHWAYS,LOCALLY ADMINISTERED ROADS,FEDERAL ROADS AND UNCLASSIFIED,TOTAL,ADMIN- ISTRATION AND MISCEL- LANEOUS,HIGHWAY LAW ENFORCEMENT AND SAFETY,INTEREST,BOND RETIREMENT (2),TOTAL DISBURSE- MENTS
0,Alabama,1082377.0,532245.0,16.0,1614638,39663.0,444070.0,NaN,483733,558335,262651.0,16130.0,113943.0,3049430
1,Alaska,862632.0,125923.0,NaN,988555,156559.0,167313.0,NaN,323872,106047,142171.0,11591.0,29427.0,1601663
2,Arizona,1248888.0,798850.0,NaN,2047738,185896.0,823197.0,NaN,1009093,814306,202285.0,111227.0,610681.0,4795330
3,Arkansas,1602089.0,226569.0,839.0,1829497,231872.0,405974.0,NaN,637846,387038,329322.0,19990.0,128860.0,3332553
4,California,6974319.0,5653622.0,NaN,12627941,2758204.0,5112729.0,NaN,7870933,1213033,5361640.0,1137770.0,1444774.0,29656091


In [106]:
#assign year to table
spending = highway_fund.iloc[:, [0, 4]].copy()
spending.columns = ["State", "capital_outlay"]
spending["year"] = 2024
spending.head()

,State,capital_outlay,year
0,Alabama,1614638,2024
1,Alaska,988555,2024
2,Arizona,2047738,2024
3,Arkansas,1829497,2024
4,California,12627941,2024


In [110]:
spending_no_total = spending[spending["State"] != "Total"]

highway_chart = alt.Chart(spending_no_total).mark_bar().encode(
    x=alt.X("State:N", sort="-y", title="State"),
    y=alt.Y("capital_outlay:Q", title="Highway Spending (Thousands of $)"),
    tooltip=["State", "capital_outlay"]
).properties(
    title="Highway Spending by State (2024)",
    width=700,
    height=350
)

highway_chart

alt.Chart(...)

**Baby names and Highway spending of 2024**

In [107]:
#joined baby names and spending year 2024
joined = pd.merge(
    baby_names,
    spending,
    on=["State", "year"],
    how="inner"
)

joined.head()

,State,rank,name,year,sex,capital_outlay
0,Alabama,1,Charlotte,2024,F,1614638
1,Alaska,1,Amelia,2024,F,988555
2,Arizona,1,Olivia,2024,F,2047738
3,Arkansas,1,Olivia,2024,F,1829497
4,California,1,Mia,2024,F,12627941


In [108]:
#create a scatter plot of Olivia's with highway spending
olivia = joined[ (joined["name"] == "Olivia") & (joined["sex"] == "F")]

chart = alt.Chart(olivia).mark_circle(size=100).encode(
    x=alt.X("capital_outlay:Q", title="Highway Spending (Thousands of $)"),
    y=alt.Y("rank:Q", scale=alt.Scale(reverse=True), title="Olivia's Rank"),
    tooltip=["State", "rank", "capital_outlay"]
).properties(
    title="Highway Spending vs Olivia's Rank (2024)",
    width=550,
    height=350
)

chart

alt.Chart(...)